# 04 · Our contribution: fine-tuning Whisper-small and wav2vec 2.0 XLS-R on our train split

Runs on Kaggle with a GPU. **Settings → Accelerator: GPU, Internet: On.** No manual uploads.
Set `SYSTEM` to `whisper_small_ft` or `xlsr_300m_ft` (one Kaggle session each).

`scripts/finetune.py` trains on the train split, chooses the checkpoint and the decoding settings
on dev, and decodes test once at the end (settings: `configs/finetune.yaml`), in a virtual
environment with the pinned packages of `requirements/finetune.txt`.
`TRIAL = True` runs 30 steps on 200 train clips and 20 dev/test clips into `/kaggle/working/trial`.

Output: `/kaggle/working/results_<SYSTEM>.zip` (predictions, `.meta.json`, training log) and the
fine-tuned weights `/kaggle/working/<SYSTEM>_best/` (not in the repository).

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
SYSTEM = "whisper_small_ft"
TRIAL = False

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; using anonymous access")

In [ ]:
# The repository and a virtual environment with only our pinned packages live in /tmp, so that the
# preinstalled packages of the Kaggle image cannot conflict and /kaggle/working holds only the results.
%cd /tmp
!rm -rf repo venv && git clone -q -b $BRANCH $REPO_URL repo
%cd /tmp/repo
!git log -1 --format='commit %H'
!pip install -q virtualenv==21.15.9 && python -m virtualenv -q /tmp/venv
PY = "/tmp/venv/bin/python"
!{PY} -m pip install -q -r requirements/finetune.txt
!{PY} -m pip freeze > /kaggle/working/pip_freeze.txt && cat /kaggle/working/pip_freeze.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"

In [ ]:
!{PY} -m pytest -q tests/test_finetune.py tests/test_inference.py

In [ ]:
!{PY} scripts/unpack_audio.py

In [ ]:
import subprocess
extra = ["--max-steps", "30", "--limit-train", "200", "--limit-eval", "20", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
r = subprocess.run([PY, "scripts/finetune.py", "--system", SYSTEM, *extra])
print("exit code", r.returncode)

In [ ]:
if TRIAL:
    !head -c 3000 /kaggle/working/trial/train_logs/$SYSTEM.json
    !head -n 3 /kaggle/working/trial/predictions/$SYSTEM.dev.jsonl
else:
    !cp -r work/models/$SYSTEM/best /kaggle/working/{SYSTEM}_best
    !zip -q -r /kaggle/working/results_{SYSTEM}.zip results/predictions/{SYSTEM}.* results/train_logs/{SYSTEM}.json
    !ls -lh /kaggle/working